In [0]:
import mlflow

# Add this diagnostic code before the download_artifacts call
client_mlflow = mlflow.MlflowClient()
mv = client_mlflow.get_model_version_by_alias(MODEL_NAME, MODEL_ALIAS)
run_id = mv.run_id

# List all artifacts for this run
artifacts = client_mlflow.list_artifacts(run_id)
print("Available artifacts:")
for artifact in artifacts:
    print(f"  - {artifact.path} (is_dir: {artifact.is_dir})")

In [0]:
# Ensure PyYAML is installed
%pip install pyyaml

import mlflow
import yaml
from pathlib import Path

# Function to download model artifacts based on MLmodel file flavors section

def download_flavor_artifact(model_name, model_alias, artifact_key):
    """
    Download an artifact referenced by MLflow model flavor using model name, alias, artifact key.
    Returns artifact content (for JSON) or local file path.
    """
    client = mlflow.MlflowClient()
    mv = client.get_model_version_by_alias(model_name, model_alias)
    model_uri = f"models:/{model_name}/{mv.version}"

    # Download MLmodel file from the model URI
    mlmodel_path = mlflow.artifacts.download_artifacts(artifact_uri=f"{model_uri}/MLmodel")
    with open(mlmodel_path, "r") as mlmodel_file:
        mlmodel_file_content = mlmodel_file.read()
        print("MLmodel raw content:\n", mlmodel_file_content[:500])
        try:
            mlmodel = yaml.safe_load(mlmodel_file_content)
        except Exception as e:
            print(f"Error: Could not parse MLmodel as YAML: {e}")
            return None

    # Navigate to desired artifact path in flavors
    flavors = mlmodel.get("flavors", {})
    pyfunc = flavors.get("python_function", {})
    artifacts = pyfunc.get("artifacts", {})
    artifact_info = artifacts.get(artifact_key, {})
    artifact_rel_path = artifact_info.get("path")
    if not artifact_rel_path:
        raise FileNotFoundError(f"Artifact {artifact_key!r} not found in MLmodel 'flavors.artifacts'.")

    # Download the artifact file
    artifact_full_path = mlflow.artifacts.download_artifacts(artifact_uri=f"{model_uri}/{artifact_rel_path}")
    # Dump and print the artifact file raw content
    with open(artifact_full_path, 'r') as f:
        artifact_file_content = f.read()
        print(f"\nArtifact ({artifact_key}) raw content:\n", artifact_file_content[:500])

    # Return JSON if file ends with .json
    if artifact_full_path.endswith('.json'):
        try:
            import json
            return json.loads(artifact_file_content)
        except Exception as e:
            print(f"Error: Artifact is not valid JSON: {e}")
            return None
    return artifact_full_path

# Example usage
MODEL_NAME = "workspace.default.llmops_support_agent"
MODEL_ALIAS = "candidate"
prompt_config = download_flavor_artifact(MODEL_NAME, MODEL_ALIAS, "prompt_config")
print(prompt_config)
